In [2]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/processed/candidate_pages.csv", header = 0)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1064 entries, 0 to 1063
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   company_id             1064 non-null   str    
 1   internal_url           1064 non-null   str    
 2   candidate_status_code  1058 non-null   float64
 3   candidate_fetch_url    1058 non-null   str    
 4   candidate_text         1027 non-null   str    
 5   candidate_error        35 non-null     str    
dtypes: float64(1), str(5)
memory usage: 50.0 KB


In [21]:
df_clean = df[(df['candidate_status_code'] != 404.0) &
              (df['candidate_text'].notna())
              ]

In [23]:
df_clean.to_csv('../data/processed/test.csv')

In [19]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 1027 entries, 0 to 1063
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   company_id             1027 non-null   str    
 1   internal_url           1027 non-null   str    
 2   candidate_status_code  1027 non-null   float64
 3   candidate_fetch_url    1027 non-null   str    
 4   candidate_text         1027 non-null   str    
 5   candidate_error        0 non-null      str    
dtypes: float64(1), str(5)
memory usage: 56.2 KB


## START CLEANING

In [30]:
import re

In [26]:
df_raw = pd.read_csv("../data/processed/test.csv", header= 0)

In [27]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1027 entries, 0 to 1026
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             1027 non-null   int64  
 1   company_id             1027 non-null   str    
 2   internal_url           1027 non-null   str    
 3   candidate_status_code  1027 non-null   float64
 4   candidate_fetch_url    1027 non-null   str    
 5   candidate_text         1027 non-null   str    
 6   candidate_error        0 non-null      float64
dtypes: float64(2), int64(1), str(4)
memory usage: 56.3 KB


In [28]:
df_raw = df_raw.drop(
    columns = [
        column
        for column in df_raw.columns
        if column.startswith('Unnamed:')
    ], errors = 'ignore'
)

In [ ]:
req_cols = {
    'company_id', 
    'internal_url', 
    'candidate_status_code', 
    'candidate_text', 
    'candidate_error'
}

missing_cols = req_cols.difference(
    df_raw.columns
)

URL_PATTERN = re.compile(
    r'(?:https?://|www\.)\S+',
    flags = re.IGNORECASE
)
EMAIL_PATTERN = re.compile(
    r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
    flags=re.IGNORECASE,
)

REPEATED_PUNCTUATION_PATTERN = re.compile(
    r"([!?.,;:])\1{2,}"
)

UNWANTED_PATH_PATTERN = re.compile(
    r"(?:^|/)(?:"
    r"contact(?:-us)?|"
    r"contacto|contato|fale-conosco|kontakt|"
    r"iletisim|nous-contacter|"
    r"blog|news|noticias|actualites|"
    r"privacy|privacy-policy|"
    r"terms|terms-and-conditions|"
    r"cookies?|login|search|cart|checkout"
    r")(?:/|$)",
    flags=re.IGNORECASE,
)


In [ ]:
import html
import unicodedata
from urllib.parse import urlparse

from country_values_distance.config import MAX_TEXT_CHARACTERS


def clean_text(val:object):
    txt = html.unescape(str(val))

    txt = unicodedata.normalize(
        "NFKC", 
        txt
    )

    cleaned_char = list[str] = []

    for char in txt:
        category = unicodedata.category(char)
        if char.isspace():
            cleaned_char.append(" ")
            continue

        if char == '\uffd':
            cleaned_char.append(' ')
            continue
        if category.startswith("C"):
            cleaned_char.append(" ")
            continue

        # Remove emoji and decorative symbols.
        if category in {"So", "Sk"}:
            cleaned_char.append(" ")
            continue

        if char in {
            "\ufe0e",
            "\ufe0f",
            "\u20e3",
        }:
            continue

        cleaned_char.append(char)

    text = ''.join(cleaned_char)
    text = URL_PATTERN.sub(' ', text)
    text = EMAIL_PATTERN.sub(' ', text)
    text = REPEATED_PUNCTUATION_PATTERN.sub(
        r"\1",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text[:MAX_TEXT_CHARACTERS]

def is_unwanted_page(val:object):
    path = urlparse(str(val)).lower().strip()
    return bool(UNWANTED_PATH_PATTERN.search(path))


def calculate_text_metrics(txt:str):
    txt_length = len(txt)
    letter_count= sum(character.isalpha() for character in txt)






